# Visualización de la Evaluación de Métodos de Extracción
Este notebook consolida los resultados de todos los métodos evaluados y permite comparar visualmente su rendimiento en los archivos de `extract-bench`.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurar estilo de Seaborn
sns.set_theme(style="whitegrid")

### 1. Carga de Datos
Iteramos sobre todos los archivos `results.csv` en la carpeta `evaluations_result/` para crear un único DataFrame.

In [ ]:
eval_dir = Path("evaluations_result")
all_data = []

for method_dir in eval_dir.iterdir():
    if method_dir.is_dir():
        csv_path = method_dir / "results.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df['Metodo'] = method_dir.name
            all_data.append(df)

if all_data:
    df_full = pd.concat(all_data, ignore_index=True)
    # Limpiar errores: Convertir "Error" a NaN y asegurar que la columna es numérica
    df_full['Puntuación'] = pd.to_numeric(df_full['Puntuación'], errors='coerce')
    # Eliminar filas con NaN si queremos analizar solo los exitosos
    df_full = df_full.dropna(subset=['Puntuación'])
    print(f"Cargados {len(df_full)} registros de evaluación en total.")
else:
    print("No se encontraron resultados.")

### 2. Puntuación Media por Método

In [ ]:
plt.figure(figsize=(10, 6))
mean_scores = df_full.groupby('Metodo')['Puntuación'].mean().reset_index().sort_values('Puntuación', ascending=False)

ax = sns.barplot(data=mean_scores, x='Puntuación', y='Metodo', hue='Metodo', palette='viridis', dodge=False, legend=False)
plt.title('Puntuación Media por Método', fontsize=16)
plt.xlabel('Puntuación Media', fontsize=12)
plt.ylabel('Método', fontsize=12)

for p in ax.patches:
    width = p.get_width()
    plt.text(width + 0.01, p.get_y() + p.get_height()/2. + 0.05, '{:1.3f}'.format(width), ha="left")

plt.xlim(0, 1.1)
plt.tight_layout()
plt.show()

### 3. Distribución de Puntuaciones (Boxplot)
Muestra la dispersión: si las cajas son pequeñas significa que el modelo es muy consistente. Si hay muchos puntos fuera (outliers) o las cajas son largas, significa que a veces saca 1 y otras 0.

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(data=df_full, x='Puntuación', y='Metodo', hue='Metodo', palette='viridis', dodge=False, legend=False)
sns.stripplot(data=df_full, x='Puntuación', y='Metodo', color='black', alpha=0.3, size=4, jitter=True)

plt.title('Distribución de Puntuaciones por Método', fontsize=16)
plt.xlabel('Puntuación', fontsize=12)
plt.ylabel('Método', fontsize=12)

plt.tight_layout()
plt.show()